# Lab 6.5 &mdash; LangChain and Chroma

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Embed and store a corpus in one call with <code>Chroma.from_documents()</code>
- Read the scores that <code>similarity_search_with_score</code> gives you and top-k does not
- Build a retriever, and choose between <code>similarity</code> and <code>mmr</code> on a corpus that needs the choice
- Scope a retriever with a metadata filter, so the chain in Lab 6.6 inherits it

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **This lab makes no gateway calls.** A retriever is not a model &mdash; it embeds,
> ranks and returns. The chat model arrives in Lab 6.6.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Ten short passages from a company handbook. Note what each one carries besides its text:
# a category, a source file and a page. Those three are what Lab 6.3 filters on and what
# Lab 6.7 cites -- metadata is not decoration, it is the half of retrieval that is exact.
#
# Note also what is NOT here: nothing mentions salary, notice period or the share price.
# Labs 6.6 and 6.8 need that gap, because refusing is a feature.

HANDBOOK = [
    {"text": "Annual leave is 24 days per year for full-time employees. Leave must be applied "
             "for at least 3 working days in advance. Unused annual leave cannot be carried "
             "forward to the next financial year.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Sick leave is 12 days per year. Notify your manager by 10 AM on the day of "
             "absence. A medical certificate is required for absences of more than 2 "
             "consecutive days.",
     "category": "leave", "source": "handbook.pdf", "page": 5},
    {"text": "Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks. Both "
             "must be applied for at least 30 days before the expected date.",
     "category": "leave", "source": "handbook.pdf", "page": 6},
    {"text": "Employees may work from home up to 3 days per week with team lead approval. "
             "Core hours are 10 AM to 4 PM IST, and you must be reachable during them.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "A VPN connection is mandatory for reaching internal systems from home. "
             "Contact the IT helpdesk for VPN setup.",
     "category": "wfh", "source": "handbook.pdf", "page": 8},
    {"text": "Internet reimbursement is 1,500 per month for employees working from home. "
             "Submit the broadband bill to finance by the 5th of each month.",
     "category": "expense", "source": "handbook.pdf", "page": 9},
    {"text": "Travel expenses must be submitted with original receipts within 7 working days "
             "of travel. The meal allowance during client visits is 500 per day.",
     "category": "expense", "source": "handbook.pdf", "page": 12},
    {"text": "Laptops are provided by the company and replaced every 3 years. Software "
             "licence requests go through the IT helpdesk and must not be bought directly.",
     "category": "tech", "source": "tech-guide.pdf", "page": 7},
    {"text": "The backend stack is Python with FastAPI, and Java with Spring Boot. New "
             "services should use Python unless there is a specific reason not to. "
             "PostgreSQL is the primary database.",
     "category": "tech", "source": "tech-guide.pdf", "page": 3},
    {"text": "The Bangalore office is the headquarters, on the 5th floor, with 200+ staff. "
             "The Mumbai office is in the Worli business district, Tower A, 12th floor.",
     "category": "office", "source": "office-directory.pdf", "page": 15},
]

print(f"{len(HANDBOOK)} passages, "
      f"{len({d['category'] for d in HANDBOOK})} categories, "
      f"{len({d['source'] for d in HANDBOOK})} source files")

In [ ]:
# ------------------------------------------------- the corpus as LangChain Documents
from langchain_core.documents import Document

def handbook_documents() -> list:
    """One Document per passage: the text, and everything else as metadata."""
    return [Document(page_content=d["text"],
                     metadata={"category": d["category"],
                               "source": d["source"],
                               "page": d["page"]})
            for d in HANDBOOK]

print(len(handbook_documents()), "Document objects")

## Concept

`Chroma.from_documents()` does three things in one call: embeds every document, writes the
vectors, text and metadata into a collection, and hands back a store object.

A **retriever** is then a thin wrapper with one job &mdash; `str` in, `list[Document]` out.
That signature is the whole reason it drops into an LCEL chain in the next lab without
ceremony.

Two dials, and both are decisions:

| | |
|---|---|
| `k` | how many chunks come back. Always exactly `k`, however bad the last one is |
| `search_type` | `similarity` takes the top k. `mmr` trades a little relevance for variety, and earns its keep when the corpus repeats itself |

## Section 1 &mdash; From Documents to a searchable store

Note what you do **not** pass: an embedding function for Chroma to guess at. You hand it
the model, so there is no ambiguity about what indexed the corpus and what will embed the
query. They must be the same model, and this is how you guarantee it.

In [ ]:
from langchain_chroma import Chroma

def build_store():
    """Embed the corpus and store it. Which model embeds it?

    It has to be the same one that will embed the queries -- a store indexed with one
    model and searched with another returns noise, and nothing raises."""
    return Chroma.from_documents(
        documents=handbook_documents(),
        embedding=BLANK,
        collection_name="handbook_lc")


_store = {}
def store():
    """Given -- build once per kernel, not once per query."""
    if "s" not in _store:
        _store["s"] = build_store()
    return _store["s"]

In [ ]:
# --- Self-check: Section 1   (a real Chroma store -- embeddings are local, no gateway)
check("the store holds every passage",
      lambda: store()._collection.count() == len(HANDBOOK))
check("similarity_search returns Documents, not strings",
      lambda: isinstance(store().similarity_search("annual leave", k=1)[0], Document))
check("a specific question finds the annual-leave passage",
      lambda: "24 days" in store().similarity_search("How much annual leave do I get?",
                                                     k=1)[0].page_content)
check("the metadata survived the round trip",
      lambda: store().similarity_search("annual leave", k=1)[0].metadata["source"] == "handbook.pdf")

def _scores():
    print("  scores are DISTANCES here -- lower is closer, unlike the cosine in Lab 6.2")
    for doc, s in store().similarity_search_with_score("How do I work from home?", k=3):
        print(f"  [{s:.4f}] ({doc.metadata['category']}) {doc.page_content[:56]}...")
guard(_scores)

## Section 2 &mdash; The retriever, and the two dials

The corpus has **three** leave passages that all look alike to an embedding model. Ask a
broad question about benefits with plain similarity and you get three variations on leave
and nothing about working from home or expenses.

That is the case `mmr` exists for.

In [ ]:
def broad_search_type() -> str:
    """A user asks "what leave can I take?". The corpus has three leave passages, and
    plain similarity hands back all three -- annual, sick and maternity -- with nothing
    else. Which search_type trades a little relevance for a wider spread?"""
    return BLANK


def leave_only() -> dict:
    """A filter that pins a retriever to leave passages, whatever the question.

    LangChain passes this through to Chroma as the `where` clause you wrote in Lab 6.3."""
    return BLANK


def make_retriever(k: int = 3, search_type: str = "similarity", flt: dict | None = None):
    """Given -- your decisions, wired in."""
    kwargs = {"k": k}
    if search_type == "mmr":
        kwargs["fetch_k"] = k * 3        # look at 3k, return a diverse k
    if flt:
        kwargs["filter"] = flt
    return store().as_retriever(search_type=search_type, search_kwargs=kwargs)

In [ ]:
# --- Self-check: Section 2   (retriever objects and what they return)
def cats(docs):
    return [d.metadata["category"] for d in docs]

check("a retriever takes a string and returns Documents",
      lambda: all(isinstance(d, Document)
                  for d in make_retriever().invoke("What is the expense policy?")))
check("k really does control how many come back",
      lambda: len(make_retriever(k=2).invoke("annual leave")) == 2)
check("you chose a search_type LangChain knows",
      lambda: broad_search_type() in ("similarity", "mmr"))
check("plain similarity answers 'what leave can I take?' with three leave passages",
      lambda: cats(make_retriever(k=3).invoke("What leave can I take?")) == ["leave"] * 3,
      "all three are relevant -- and the user learns nothing they could not have guessed")
check("your choice returns more than one category for the same question",
      lambda: len(set(cats(make_retriever(k=3, search_type=broad_search_type())
                           .invoke("What leave can I take?")))) > 1,
      "similarity keeps handing back the same topic; mmr is the dial for that")
check("the filtered retriever never leaves the leave passages",
      lambda: set(cats(make_retriever(k=3, flt=leave_only())
                       .invoke("What can I claim for travel?"))) == {"leave"},
      "note the question is about expenses and it STILL only returns leave")

def _compare():
    for label, st in (("similarity", "similarity"), (broad_search_type(), broad_search_type())):
        got = cats(make_retriever(k=3, search_type=st).invoke("What leave can I take?"))
        print(f"  {label:11} -> {got}")
    print("  ...and read what that cost: two of the three LEAVE passages are gone.")
    print("  mmr bought variety by giving up relevance. On this question that is")
    print("  probably a bad trade -- which is the point. It is a dial, not a fix.")
guard(_compare)

In [ ]:
score()

## Your turn

1. The filtered retriever answered a travel-expenses question with leave passages and did
   not complain. Build the same retriever with `k=3` and print the scores. Is there a
   distance at which you would rather return nothing? Write that rule down &mdash; Lab 6.6
   makes the model act on it.
2. Raise `fetch_k` on the mmr retriever from `3k` to `10k`. Where does the spread stop
   improving, and what does it cost?
3. Build a *second* store from the same documents with `collection_name="handbook_alt"` but
   chunked differently, and ask both the same question. Two stores, one corpus, different
   answers &mdash; which is the honest version of &ldquo;we upgraded retrieval&rdquo;.